In [ ]:
# Cell 1: Install dependencies and LocalTunnel for web tunneling
!pip install --quiet tensorflow streamlit pyngrok seaborn matplotlib scikit-learn pillow
!npm install -g localtunnel

In [ ]:
# (Optional) Mount Google Drive if your dataset is in Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/dataset.zip -d dataset/

In [ ]:
# Cell 2: Generate sample dataset structure
import os
import numpy as np
from PIL import Image, ImageDraw

def create_sample_dataset(base_dir="dataset"):
    sizes = {
        "S":  {"chest_w": 40, "torso_h": 65, "sleeve_l": 20},
        "M":  {"chest_w": 52, "torso_h": 78, "sleeve_l": 26},
        "L":  {"chest_w": 64, "torso_h": 90, "sleeve_l": 32},
        "XL": {"chest_w": 76, "torso_h": 102, "sleeve_l": 38}
    }

    for split, count in [("train", 120), ("test", 30)]:
        for label, dims in sizes.items():
            folder = os.path.join(base_dir, split, label)
            os.makedirs(folder, exist_ok=True)
            for i in range(count):
                # Add natural dimension jittering
                cw = dims["chest_w"] + np.random.randint(-3, 4)
                th = dims["torso_h"] + np.random.randint(-4, 5)
                sl = dims["sleeve_l"] + np.random.randint(-2, 3)

                # Create flat-lay garment silhouette on neutral background
                bg_color = (np.random.randint(230, 255), np.random.randint(230, 255), np.random.randint(230, 255))
                cloth_color = (np.random.randint(20, 180), np.random.randint(20, 180), np.random.randint(20, 180))

                img = Image.new("RGB", (128, 128), color=bg_color)
                draw = ImageDraw.Draw(img)

                cx, cy = 64, 64
                # Torso
                draw.rectangle([cx - cw // 2, cy - th // 2, cx + cw // 2, cy + th // 2], fill=cloth_color)
                # Left Sleeve
                draw.polygon([(cx - cw // 2, cy - th // 2), (cx - cw // 2 - sl, cy - th // 2 + 18), (cx - cw // 2, cy - th // 2 + 30)], fill=cloth_color)
                # Right Sleeve
                draw.polygon([(cx + cw // 2, cy - th // 2), (cx + cw // 2 + sl, cy - th // 2 + 18), (cx + cw // 2, cy - th // 2 + 30)], fill=cloth_color)

                img.save(os.path.join(folder, f"{label}_{i}.png"))

    print("[SUCCESS] Dataset initialized under 'dataset/train' and 'dataset/test'.")

create_sample_dataset()

In [ ]:
# Cell 3: Build & Train the CNN Model
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 4
CLASS_NAMES = ['L', 'M', 'S', 'XL']
TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"
MODEL_OUTPUT_PATH = "cloth_size_model.keras"

# 1. Pipeline loaders
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=42
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

autotune = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=autotune)
test_ds = test_ds.cache().prefetch(buffer_size=autotune)

# 2. CNN Architecture
def build_model():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.Rescaling(1.0 / 255.0),

        # Conv Block 1
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # Conv Block 2
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # Conv Block 3
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # Dense Head
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])
    return model

model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# 3. Training
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    verbose=1
)

# 4. Evaluation
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# 5. Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Cloth Size Confusion Matrix")
plt.xlabel("Predicted Size")
plt.ylabel("True Size")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200)
plt.show()

# Save final weights
model.save(MODEL_OUTPUT_PATH)
print(f"[SUCCESS] Saved model to '{MODEL_OUTPUT_PATH}'.")# Cell 3: Build & Train the CNN Model
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 4
CLASS_NAMES = ['L', 'M', 'S', 'XL']
TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"
MODEL_OUTPUT_PATH = "cloth_size_model.keras"

# 1. Pipeline loaders
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=42
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

autotune = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=autotune)
test_ds = test_ds.cache().prefetch(buffer_size=autotune)

# 2. CNN Architecture
def build_model():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.Rescaling(1.0 / 255.0),

        # Conv Block 1
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # Conv Block 2
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # Conv Block 3
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # Dense Head
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])
    return model

model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# 3. Training
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    verbose=1
)

# 4. Evaluation
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# 5. Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Cloth Size Confusion Matrix")
plt.xlabel("Predicted Size")
plt.ylabel("True Size")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200)
plt.show()

# Save final weights
model.save(MODEL_OUTPUT_PATH)
print(f"[SUCCESS] Saved model to '{MODEL_OUTPUT_PATH}'.")# Cell 3: Build & Train the CNN Model
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 4
CLASS_NAMES = ['L', 'M', 'S', 'XL']
TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"
MODEL_OUTPUT_PATH = "cloth_size_model.keras"

# 1. Pipeline loaders
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=42
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

autotune = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=autotune)
test_ds = test_ds.cache().prefetch(buffer_size=autotune)

# 2. CNN Architecture
def build_model():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.Rescaling(1.0 / 255.0),

        # Conv Block 1
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # Conv Block 2
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # Conv Block 3
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # Dense Head
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])
    return model

model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# 3. Training
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    verbose=1
)

# 4. Evaluation
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# 5. Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Cloth Size Confusion Matrix")
plt.xlabel("Predicted Size")
plt.ylabel("True Size")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200)
plt.show()

# Save final weights
model.save(MODEL_OUTPUT_PATH)
print(f"[SUCCESS] Saved model to '{MODEL_OUTPUT_PATH}'.")

In [ ]:
# Cell 4: Write app.py
%%writefile app.py
import os
import streamlit as st
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow import keras

st.set_page_config(page_title="Cloth Size Predictor", page_icon="👕", layout="centered")

MODEL_PATH = "cloth_size_model.keras"
IMAGE_SIZE = (128, 128)

CLASS_LABELS = {
    0: ("Large (L)", "Chest width ~40-42 in / 102-107 cm"),
    1: ("Medium (M)", "Chest width ~38-40 in / 96-101 cm"),
    2: ("Small (S)", "Chest width ~35-37 in / 89-94 cm"),
    3: ("Extra Large (XL)", "Chest width ~43-45 in / 109-114 cm")
}

@st.cache_resource
def get_model():
    if not os.path.exists(MODEL_PATH):
        return None
    return keras.models.load_model(MODEL_PATH)

st.title("👕 AI Cloth Size Predictor")
st.write("Convolutional Neural Network classification for clothing flat-lays.")
st.markdown("---")

model = get_model()
if model is None:
    st.error("Model weights not found. Please train the model first.")
    st.stop()

uploaded_file = st.file_uploader("Upload clothing photo (JPG, PNG)", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    col1, col2 = st.columns([1, 1], gap="medium")

    pil_img = Image.open(uploaded_file).convert("RGB")
    resized_img = pil_img.resize(IMAGE_SIZE)
    img_array = np.expand_dims(np.array(resized_img, dtype=np.float32), axis=0)

    with col1:
        st.subheader("Garment Preview")
        st.image(pil_img, use_column_width=True)

    with col2:
        st.subheader("Prediction")
        if st.button("🔍 Predict Size", type="primary", use_container_width=True):
            with st.spinner("Classifying..."):
                preds = model.predict(img_array, verbose=0)[0]
                idx = int(np.argmax(preds))
                confidence = float(preds[idx]) * 100.0
                size_name, fit_note = CLASS_LABELS[idx]

            st.metric(label="Predicted Category", value=size_name, delta=f"{confidence:.2f}% Confidence")
            st.info(f"**Fit Guidance:** {fit_note}")
            st.progress(confidence / 100.0)

            chart_data = {CLASS_LABELS[i][0]: float(preds[i] * 100.0) for i in range(len(CLASS_LABELS))}
            st.bar_chart(chart_data)

In [ ]:
# Cell 5: Launch Streamlit and tunnel to a public URL
import urllib

# 1. Print your external Colab IP (This is your LocalTunnel password)
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("=" * 60)
print(f"👉 YOUR LOCALTUNNEL PASSWORD / TUNNEL IP: {public_ip}")
print("=" * 60)

# 2. Run Streamlit in background and tunnel with LocalTunnel
!streamlit run app.py & npx localtunnel --port 8501

👉 YOUR LOCALTUNNEL PASSWORD / TUNNEL IP: 34.143.135.136
/bin/bash: line 1: streamlit: command not found
⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 

In [ ]:
# Cell 5: Launch Streamlit and tunnel to a public URL
import urllib

# 1. Print your external Colab IP (This is your LocalTunnel password)
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("=" * 60)
print(f"👉 YOUR LOCALTUNNEL PASSWORD / TUNNEL IP: {public_ip}")
print("=" * 60)

# 2. Run Streamlit in background and tunnel with LocalTunnel
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
# 1. Terminate any previous background tunnels/streamlit instances
!pkill -f streamlit
!pkill -f cloudflared

# 2. Download and install Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 3. Launch Streamlit with CORS/XSRF flags set for external tunneling
!nohup streamlit run app.py \
    --server.port 8501 \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    --server.headless true > /content/streamlit.log 2>&1 &

import time
time.sleep(3)

# 4. Start Cloudflare Tunnel
!nohup cloudflared tunnel --url http://127.0.0.1:8501 --no-autoupdate > /content/tunnel.log 2>&1 &
time.sleep(5)

# 5. Print the public URL
print("\n" + "=" * 60)
!grep -o 'https://[-0-9a-z]*\.trycloudflare.com' /content/tunnel.log | head -n 1
print("=" * 60)


https://plains-reads-conversation-immediate.trycloudflare.com


In [ ]:
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow import keras

# 1. Load the trained model
model = keras.models.load_model("cloth_size_model.keras")
class_labels = {0: "Large (L)", 1: "Medium (M)", 2: "Small (S)", 3: "Extra Large (XL)"}

# 2. Pick a test image from any class folder
test_image_path = "dataset/test/S/S_0.png"  # Replace with any image path

img = Image.open(test_image_path).convert("RGB")
img_resized = img.resize((128, 128))
img_array = np.expand_dims(np.array(img_resized, dtype=np.float32), axis=0)

# 3. Predict
probs = model.predict(img_array, verbose=0)[0]
predicted_idx = int(np.argmax(probs))
confidence = probs[predicted_idx] * 100

print(f"File: {test_image_path}")
print(f"Predicted Size: {class_labels[predicted_idx]}")
print(f"Confidence: {confidence:.2f}%")
print("Probabilities per class:", {class_labels[i]: f"{probs[i]*100:.1f}%" for i in range(4)})